# Camada Gold — physical_vendas_caixa
Lê a Silver, aplica as **regras de negócio 6 a 10 do regras.xlsx** e publica cada tabela em
**Delta (gold/)** e no **SQL Server**, sempre com **MERGE (upsert)**. No fim, valida e exporta CSVs.

| Regra | Tabela | Chave (grão) |
|---|---|---|
| 6 — Receita por loja por mês (MoM) | `gold_receita_loja_mes` | id_loja, ano, mes |
| 7 — Ticket médio por loja e pagamento (anual) | `gold_ticket_medio_loja_pagamento_ano` | id_loja, ano, tipo_pagamento |
| 8 — % vendas com/sem CPF por loja por mês | `gold_cpf_identificado_loja_mes` | id_loja, ano, mes |
| 9 — Transações por caixa por dia | `gold_transacoes_caixa_dia` | data_venda, id_loja, id_caixa |
| 10 — Feriados nacionais (`e_feriado`) | `gold_vendas_dia_feriado` | data_venda |
| Apoio — cadastro de lojas | `gold_dim_lojas` | id_loja |



In [0]:
%pip install python-dotenv azure-storage-file-datalake azure-identity

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuração

In [0]:
import os
import shutil
from datetime import date, timedelta

from dotenv import load_dotenv
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

def carregar_env():
    """Procura o .env: ENV_PATH (se definido), a pasta do notebook e as pastas acima dela."""
    candidatos = []
    if os.getenv("ENV_PATH"):
        candidatos.append(os.environ["ENV_PATH"])
    try:
        nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        pasta = os.path.dirname(f"/Workspace{nb}")
        for _ in range(4):                         # pasta do notebook e até 3 níveis acima
            candidatos.append(os.path.join(pasta, ".env"))
            pasta = os.path.dirname(pasta)
    except Exception:
        pass
    candidatos.append(os.path.join(os.getcwd(), ".env"))
    for c in candidatos:
        if os.path.isfile(c):
            load_dotenv(c, override=True)
            print(".env carregado de:", c)
            return
    print("[aviso] .env não encontrado. Procurado em:")
    for c in candidatos:
        print("   ", c)


carregar_env()


def env(*nomes, padrao=None):
    """Primeiro nome definido no .env (aceita SQL_HOST ou SQLSERVER_HOST, por exemplo)."""
    for n in nomes:
        if os.getenv(n):
            return os.environ[n]
    return padrao


OBRIGATORIAS = {
    "STORAGE_ACCOUNT_NAME": ("STORAGE_ACCOUNT_NAME",),
    "ADLS_CLIENT_ID": ("ADLS_CLIENT_ID",),
    "ADLS_TENANT_ID": ("ADLS_TENANT_ID",),
    "ADLS_CLIENT_SECRET": ("ADLS_CLIENT_SECRET",),
    "SQL_HOST": ("SQL_HOST", "SQLSERVER_HOST"),
    "SQL_DATABASE": ("SQL_DATABASE", "SQLSERVER_DATABASE"),
    "SQL_USERNAME": ("SQL_USERNAME", "SQL_USER", "SQLSERVER_USER"),
    "SQL_PASSWORD": ("SQL_PASSWORD", "SQLSERVER_PASSWORD"),
}
faltando = [k for k, nomes in OBRIGATORIAS.items() if not env(*nomes)]
if faltando:
    raise RuntimeError(f"Variáveis ausentes no ambiente: {faltando}. Defina ENV_PATH com o caminho do .env "
                       "(ex.: os.environ['ENV_PATH'] = '/Workspace/Users/<voce>/projeto/.env') ou confira os nomes.")

# --- ADLS ---
STORAGE_ACCOUNT = os.environ["STORAGE_ACCOUNT_NAME"]
CONTAINER = os.getenv("ADLS_CONTAINER", "raw")
SILVER_CONTAINER = os.getenv("ADLS_SILVER_CONTAINER", CONTAINER)
GOLD_CONTAINER = os.getenv("ADLS_GOLD_CONTAINER", CONTAINER)
SILVER_TABLE_PATH = os.getenv("SILVER_TABLE_PATH", "silver/physical_vendas_caixa")

# --- Lojas (lidas direto da raw: regra 2) ---
# Mesmo padrão da Silver: LOJAS_REL (opcional) = caminho dentro do container raw.
# Sem ele, o notebook procura no container raw algo com "physical_lojas" no nome e detecta o formato.
LOJAS_REL = env("LOJAS_REL", "LOJAS_PATH")

# --- Regras ---
TIPOS_PAGAMENTO_VALIDOS = ["Cartão", "Dinheiro", "Pix"]          # regra 4
FATOR_SOBRECARGA = float(os.getenv("FATOR_SOBRECARGA", "3"))     # regra 9

# --- SQL Server ---
SQL_HOST = env("SQL_HOST", "SQLSERVER_HOST")
SQL_PORT = env("SQL_PORT", "SQLSERVER_PORT", padrao="1433")
SQL_DB = env("SQL_DATABASE", "SQLSERVER_DATABASE")
SQL_USER = env("SQL_USERNAME", "SQL_USER", "SQLSERVER_USER")
SQL_PASSWORD = env("SQL_PASSWORD", "SQLSERVER_PASSWORD")
SQL_SCHEMA = env("SQL_SCHEMA", "SQLSERVER_SCHEMA", padrao="squad3")
GRAVAR_SQLSERVER = os.getenv("GRAVAR_SQLSERVER", "true").lower() == "true"

# --- Exportação CSV ---
EXPORTAR_CSV = os.getenv("EXPORTAR_CSV", "true").lower() == "true"
CSV_DIR = os.getenv("CSV_DIR", "/Volumes/workspace/default/exportacoes")


def abfss(container: str, path: str) -> str:
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"


SILVER_PATH = abfss(SILVER_CONTAINER, SILVER_TABLE_PATH)
GOLD_BASE = abfss(GOLD_CONTAINER, "gold")

# Serverless só permite ler/gravar com o conector nativo "sqlserver" (não o "jdbc" genérico)
# e não executa comandos T-SQL soltos. Por isso o SQL Server é sincronizado com a Delta (ver abaixo).
SQL_OPTS = {"host": SQL_HOST, "port": SQL_PORT, "database": SQL_DB,
            "user": SQL_USER, "password": SQL_PASSWORD}

# --- Autenticação OAuth (service principal) ---
# Mesmo padrão da Silver: as credenciais vão como OPTIONS em cada leitura/escrita (funciona no serverless).
# Num cluster clássico também dá para registrá-las na sessão, o que habilita o MERGE INTO em SQL.
suffix = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
AUTH = {
    f"fs.azure.account.auth.type.{suffix}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{suffix}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{suffix}": os.environ["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{suffix}": os.environ["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{suffix}":
        f"https://login.microsoftonline.com/{os.environ['ADLS_TENANT_ID']}/oauth2/token",
}
try:
    for k, v in AUTH.items():
        spark.conf.set(k, v)
    AUTH_NA_SESSAO = True
    print("Autenticação: credenciais na sessão (cluster) · Delta com MERGE INTO")
except Exception as e:
    if "CONFIG_NOT_AVAILABLE" not in str(e):
        raise
    AUTH_NA_SESSAO = False
    print("Autenticação: credenciais por leitura/escrita (serverless) · Delta com upsert por chave")


def ler_delta(path: str) -> DataFrame:
    return spark.read.format("delta").options(**AUTH).load(path)


print("Silver:", SILVER_PATH)
print("Gold:  ", GOLD_BASE)
print("SQL:   ", f"{SQL_DB}.{SQL_SCHEMA}")

.env carregado de: /Workspace/Users/eng.stephanni@gmail.com/.env
Autenticação: credenciais por leitura/escrita (serverless) · Delta com upsert por chave
Silver: abfss://raw@internshipdatalake.dfs.core.windows.net/silver/physical_vendas_caixa
Gold:   abfss://raw@internshipdatalake.dfs.core.windows.net/gold
SQL:    internshipDatabase.squad3


## 1. Funções de publicação (MERGE no Delta e no SQL Server)

In [0]:
def tabela_delta_existe(path: str):
    """Devolve as colunas da tabela Delta, ou None se ela ainda não existe."""
    try:
        return ler_delta(path).columns
    except Exception as e:
        if "PATH_NOT_FOUND" in str(e) or "Path does not exist" in str(e) or "is not a Delta table" in str(e):
            return None
        raise


def merge_delta(df: DataFrame, nome: str, chaves: list, particoes: list) -> None:
    path = f"{GOLD_BASE}/{nome}"
    colunas = tabela_delta_existe(path)
    writer = df.write.format("delta").options(**AUTH)
    if particoes:
        writer = writer.partitionBy(*particoes)

    if colunas is None:
        writer.save(path)
        print(f"  [Delta] {nome}: criada")
        return
    if sorted(colunas) != sorted(df.columns):
        # esquema mudou (coluna nova/removida): a tabela é recalculada inteira, então reescreve
        writer.mode("overwrite").option("overwriteSchema", "true").save(path)
        print(f"  [Delta] {nome}: esquema alterado, tabela reescrita")
        return

    if not AUTH_NA_SESSAO:
        # Serverless: o MERGE INTO em SQL não recebe as credenciais por options. Upsert equivalente:
        # mantém as linhas atuais cuja chave não veio no lote e grava o lote inteiro por cima.
        atual = ler_delta(path)
        cond_chave = [atual[k].eqNullSafe(df[k]) for k in chaves]
        mantidas = atual.join(df, cond_chave, "left_anti")
        resultado = mantidas.unionByName(df)
        n_mantidas, n_lote = mantidas.count(), df.count()   # conta antes de sobrescrever
        w = resultado.write.format("delta").options(**AUTH).mode("overwrite")
        if particoes:
            w = w.partitionBy(*particoes)
        w.save(path)
        print(f"  [Delta] {nome}: upsert por chave ok ({n_mantidas:,} mantidas + {n_lote:,} do lote)")
        return

    view = f"_origem_{nome}"
    df.createOrReplaceTempView(view)
    cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in chaves)
    spark.sql(f"""
        MERGE INTO delta.`{path}` AS t
        USING {view} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  [Delta] {nome}: MERGE INTO ok")


def ler_sqlserver(query: str) -> DataFrame:
    return spark.read.format("sqlserver").options(**SQL_OPTS).option("query", query).load()


def colunas_sqlserver(tabela: str):
    try:
        linhas = ler_sqlserver(f"""SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                                   WHERE TABLE_SCHEMA = '{SQL_SCHEMA}' AND TABLE_NAME = '{tabela}'""").collect()
        return [l[0] for l in linhas] or None
    except Exception:
        return None


def sincronizar_sqlserver(nome: str) -> None:
    """Deixa a tabela do SQL Server igual à tabela Delta (que acabou de receber o merge/upsert).

    No serverless não dá para rodar MERGE em T-SQL. Como a Delta já tem o resultado do upsert por chave,
    gravar a tabela Delta inteira no SQL Server dá o mesmo estado final de um MERGE:
    - mesmo esquema: TRUNCATE + INSERT (mantém a tabela, permissões e conexões do Looker);
    - tabela nova ou esquema diferente: recria a tabela.
    """
    df = ler_delta(f"{GOLD_BASE}/{nome}")
    colunas = colunas_sqlserver(nome)
    mesmo_esquema = colunas is not None and sorted(c.lower() for c in colunas) == sorted(c.lower() for c in df.columns)
    (df.write.format("sqlserver").options(**SQL_OPTS)
        .option("dbtable", f"{SQL_SCHEMA}.{nome}")
        .option("truncate", "true" if mesmo_esquema else "false")
        .mode("overwrite").save())
    acao = "sincronizada com a Delta" if mesmo_esquema else ("criada" if colunas is None else "esquema alterado, recriada")
    print(f"  [SQL Server] {SQL_SCHEMA}.{nome}: {acao}")


PUBLICADAS = {}   # nome -> chaves, para a validação


def publicar(df: DataFrame, nome: str, chaves: list, particoes: list = None) -> None:
    print(f"\n{nome}  (chave: {', '.join(chaves)})")
    df = df.withColumn("gold_processed_at", F.current_timestamp())
    merge_delta(df, nome, chaves, particoes or [])
    if GRAVAR_SQLSERVER:
        sincronizar_sqlserver(nome)
    PUBLICADAS[nome] = chaves

## 2. Leitura da Silver e das lojas + checagem defensiva (regras 2, 3 e 4)

In [0]:
try:
    silver = ler_delta(SILVER_PATH)
except Exception as e:
    if "PATH_NOT_FOUND" in str(e) or "Path does not exist" in str(e):
        raise RuntimeError(f"Silver não encontrada em {SILVER_PATH}. Ajuste ADLS_SILVER_CONTAINER / "
                           "SILVER_TABLE_PATH no .env ou rode o notebook da Silver antes.") from None
    raise

# Adapta os nomes de coluna da Silver ao que a Gold usa (sem alterar a Silver)
print("Colunas da Silver:", silver.columns)
if "data_venda" not in silver.columns:
    if "dt_venda" not in silver.columns:
        raise RuntimeError("A Silver não tem data_venda nem dt_venda.")
    silver = silver.withColumn("data_venda", F.to_date("dt_venda"))
if "ano" not in silver.columns:
    silver = silver.withColumn("ano", F.year("data_venda"))
if "mes" not in silver.columns:
    silver = silver.withColumn("mes", F.month("data_venda"))
if "cliente_identificado" not in silver.columns:
    # CPF válido = 11 dígitos depois de tirar pontuação (mesma regra da Silver v1)
    cpf_limpo = F.regexp_replace(F.coalesce(F.col("cpf_cliente").cast("string"), F.lit("")), "[^0-9]", "")
    silver = silver.withColumn("cliente_identificado", F.length(cpf_limpo) == 11)
else:
    silver = silver.withColumn("cliente_identificado", F.coalesce(F.col("cliente_identificado").cast("boolean"), F.lit(False)))
faltam = [c for c in ["id_transacao", "id_loja", "id_caixa", "valor_total_venda", "tipo_pagamento"] if c not in silver.columns]
if faltam:
    raise RuntimeError(f"Colunas esperadas que não estão na Silver: {faltam}. Colunas disponíveis: {silver.columns}")

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

adls = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=ClientSecretCredential(os.environ["ADLS_TENANT_ID"], os.environ["ADLS_CLIENT_ID"],
                                      os.environ["ADLS_CLIENT_SECRET"]),
)
PASTAS_IGNORADAS = ("bronze", "silver", "gold")


def listar_pasta(fs, rel):
    """Arquivos dentro de rel; lista vazia se rel for um arquivo (ou não existir)."""
    try:
        return [p.name for p in fs.get_paths(path=rel, recursive=True)]
    except Exception:
        return []


def localizar_lojas():
    """Devolve (caminho relativo, formato) da physical_lojas no container raw."""
    fs = adls.get_file_system_client(CONTAINER)
    if LOJAS_REL:
        candidatos = [(LOJAS_REL.strip("/"), None)]
    else:
        candidatos = [(p.name, p.is_directory) for p in fs.get_paths(recursive=True)
                      if "physical_lojas" in p.name.lower()
                      and not p.name.lower().startswith(PASTAS_IGNORADAS)]
        if not candidatos:
            raise RuntimeError(f"Nada com 'physical_lojas' no nome no container {CONTAINER}. Defina LOJAS_REL no .env.")
        # prefere a pasta/arquivo mais "alto" (ex.: a pasta Delta e não um arquivo dentro dela)
        candidatos.sort(key=lambda c: (c[0].count("/"), c[0]))
    rel, _ = candidatos[0]
    nomes = listar_pasta(fs, rel)
    if any("_delta_log" in n for n in nomes):
        return rel, "delta"
    alvo = rel if not nomes else next((n for n in nomes if n.lower().endswith((".csv", ".parquet", ".json"))), rel)
    ext = alvo.lower().rsplit(".", 1)[-1]
    formato = {"csv": "csv", "parquet": "parquet", "json": "json"}.get(ext, "parquet")
    return rel, formato


def separador_csv(rel):
    """Lê o começo do primeiro CSV para saber se o separador é ';' ou ','."""
    fs = adls.get_file_system_client(CONTAINER)
    arq = next((n for n in listar_pasta(fs, rel) if n.lower().endswith(".csv")), rel)
    inicio = fs.get_file_client(arq).download_file(offset=0, length=4096).readall().decode("utf-8", "ignore")
    primeira = inicio.splitlines()[0] if inicio else ""
    return ";" if primeira.count(";") > primeira.count(",") else ","


LOJAS_REL_ENCONTRADO, LOJAS_FORMATO = localizar_lojas()
LOJAS_FULL_PATH = abfss(CONTAINER, LOJAS_REL_ENCONTRADO)
reader = spark.read.format(LOJAS_FORMATO).options(**AUTH)
if LOJAS_FORMATO == "csv":
    reader = (reader.option("header", "true").option("inferSchema", "true")
              .option("sep", separador_csv(LOJAS_REL_ENCONTRADO)).option("encoding", "UTF-8"))
lojas = reader.load(LOJAS_FULL_PATH).dropDuplicates(["id_loja"])
print(f"Lojas: {LOJAS_FULL_PATH} ({LOJAS_FORMATO}) · {lojas.count()} lojas")

# As regras técnicas são da Silver; aqui só conferimos. Linha fora das regras fica fora da Gold, com log.
base = (
    silver.join(lojas.select("id_loja", F.lit(True).alias("_loja_existe")), "id_loja", "left")
    .withColumn("_motivo",
        F.when(F.col("_loja_existe").isNull(), "regra 2: id_loja inexistente")
         .when(F.col("valor_total_venda").isNull() | (F.col("valor_total_venda") <= 0), "regra 3: valor <= 0")
         .when(F.col("tipo_pagamento").isNull() | ~F.col("tipo_pagamento").isin(TIPOS_PAGAMENTO_VALIDOS),
               "regra 4: tipo_pagamento fora do domínio"))
)
rejeitadas = (base.filter(F.col("_motivo").isNotNull())
              .groupBy("_motivo").agg(F.count("*").alias("linhas"),
                                      F.round(F.sum("valor_total_venda"), 2).alias("valor")).collect())
if rejeitadas:
    print("[aviso] linhas fora das regras técnicas, excluídas da Gold (corrigir na Silver):")
    for r in rejeitadas:
        print(f"   {r['_motivo']}: {r['linhas']} linhas, R$ {r['valor']}")
else:
    print("Silver ok: nenhuma violação das regras 2, 3 e 4.")

vendas = base.filter(F.col("_motivo").isNull()).drop("_loja_existe", "_motivo")

Colunas da Silver: ['id_transacao', 'id_loja', 'id_caixa', 'id_operador', 'dt_venda', 'valor_total_venda', 'cpf_cliente', 'tipo_pagamento', 'bronze_source_file', 'bronze_ingested_at', 'ano', 'mes', 'silver_processed_at']
Lojas: abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_lojas.csv (csv) · 29 lojas
Silver ok: nenhuma violação das regras 2, 3 e 4.


## 3. Calendário de feriados nacionais (regra 10)

In [0]:
def pascoa(ano: int) -> date:
    """Domingo de Páscoa (algoritmo de Meeus/Butcher)."""
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = ((h + l - 7 * m + 114) % 31) + 1
    return date(ano, mes, dia)


def feriados_nacionais(ano: int):
    """Feriados nacionais oficiais + pontos facultativos nacionais (Carnaval e Corpus Christi)."""
    p = pascoa(ano)
    oficiais = [
        (date(ano, 1, 1), "Confraternização Universal"),
        (p - timedelta(days=2), "Sexta-feira Santa"),
        (date(ano, 4, 21), "Tiradentes"),
        (date(ano, 5, 1), "Dia do Trabalho"),
        (date(ano, 9, 7), "Independência do Brasil"),
        (date(ano, 10, 12), "Nossa Senhora Aparecida"),
        (date(ano, 11, 2), "Finados"),
        (date(ano, 11, 15), "Proclamação da República"),
        (date(ano, 12, 25), "Natal"),
    ]
    if ano >= 2024:   # Lei 14.759/2023
        oficiais.append((date(ano, 11, 20), "Dia Nacional de Zumbi e da Consciência Negra"))
    facultativos = [
        (p - timedelta(days=48), "Carnaval (segunda)"),
        (p - timedelta(days=47), "Carnaval (terça)"),
        (p + timedelta(days=60), "Corpus Christi"),
    ]
    return [(d, n, True, False) for d, n in oficiais] + [(d, n, False, True) for d, n in facultativos]


lim = vendas.agg(F.min("data_venda").alias("ini"), F.max("data_venda").alias("fim")).first()
linhas_fer = [l for a in range(lim["ini"].year, lim["fim"].year + 1) for l in feriados_nacionais(a)]
feriados = spark.createDataFrame(linhas_fer, "data_venda date, nome_feriado string, "
                                             "e_feriado boolean, e_ponto_facultativo boolean")

calendario = (
    spark.sql(f"SELECT explode(sequence(DATE'{lim['ini']}', DATE'{lim['fim']}', INTERVAL 1 DAY)) AS data_venda")
    .join(feriados, "data_venda", "left")
    .withColumn("e_feriado", F.coalesce("e_feriado", F.lit(False)))
    .withColumn("e_ponto_facultativo", F.coalesce("e_ponto_facultativo", F.lit(False)))
    .withColumn("dia_semana_num", F.dayofweek("data_venda"))                 # 1 = domingo
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(d) for d in
                ["domingo", "segunda", "terça", "quarta", "quinta", "sexta", "sábado"]]), F.col("dia_semana_num")))
    .withColumn("e_fim_de_semana", F.col("dia_semana_num").isin(1, 7))
    .withColumn("tipo_dia", F.when(F.col("e_feriado"), "feriado")
                             .when(F.col("e_ponto_facultativo"), "ponto facultativo")
                             .when(F.col("e_fim_de_semana"), "fim de semana")
                             .otherwise("dia útil"))
)
print(f"Período: {lim['ini']} a {lim['fim']} · feriados nacionais: "
      f"{calendario.filter('e_feriado').count()} · pontos facultativos: {calendario.filter('e_ponto_facultativo').count()}")

Período: 2024-01-01 a 2026-08-31 · feriados nacionais: 24 · pontos facultativos: 9


## 4. Tabelas Gold

In [0]:
# Apoio: cadastro de lojas (todas as colunas da physical_lojas)
publicar(lojas, "gold_dim_lojas", ["id_loja"])


gold_dim_lojas  (chave: id_loja)
  [Delta] gold_dim_lojas: upsert por chave ok (0 mantidas + 29 do lote)
  [SQL Server] squad3.gold_dim_lojas: esquema alterado, recriada


In [0]:
# REGRA 6: Receita total por loja por mês (MoM por loja)
w_loja = Window.partitionBy("id_loja").orderBy("periodo")
receita_loja_mes = (
    vendas.groupBy("id_loja", "ano", "mes")
    .agg(F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
         F.count("*").alias("qtd_transacoes"))
    .withColumn("periodo", F.make_date("ano", "mes", F.lit(1)))
    .withColumn("_periodo_ant", F.lag("periodo").over(w_loja))
    .withColumn("_receita_ant", F.lag("receita_total").over(w_loja))
    # só compara com o mês imediatamente anterior (se a loja não vendeu nele, MoM fica nulo)
    .withColumn("receita_mes_anterior",
                F.when(F.col("_periodo_ant") == F.add_months("periodo", -1), F.col("_receita_ant")))
    .withColumn("variacao_mom_valor", F.round(F.col("receita_total") - F.col("receita_mes_anterior"), 2))
    .withColumn("variacao_mom_pct",
                F.round((F.col("receita_total") - F.col("receita_mes_anterior")) / F.col("receita_mes_anterior") * 100, 2))
    .drop("_periodo_ant", "_receita_ant")
)
publicar(receita_loja_mes, "gold_receita_loja_mes", ["id_loja", "ano", "mes"], ["ano"])


gold_receita_loja_mes  (chave: id_loja, ano, mes)
  [Delta] gold_receita_loja_mes: esquema alterado, tabela reescrita
  [SQL Server] squad3.gold_receita_loja_mes: esquema alterado, recriada


In [0]:
# REGRA 7: Ticket médio por loja e por tipo de pagamento (anual)
ticket_ano = (
    vendas.groupBy("id_loja", "ano", "tipo_pagamento")
    .agg(F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
         F.count("*").alias("qtd_transacoes"))
    .withColumn("ticket_medio", F.round(F.col("receita_total") / F.col("qtd_transacoes"), 2))
)
publicar(ticket_ano, "gold_ticket_medio_loja_pagamento_ano", ["id_loja", "ano", "tipo_pagamento"], ["ano"])


gold_ticket_medio_loja_pagamento_ano  (chave: id_loja, ano, tipo_pagamento)
  [Delta] gold_ticket_medio_loja_pagamento_ano: esquema alterado, tabela reescrita
  [SQL Server] squad3.gold_ticket_medio_loja_pagamento_ano: esquema alterado, recriada


In [0]:
# REGRA 8: % de vendas com CPF identificado vs sem CPF por loja por mês
cpf_mes = (
    vendas.groupBy("id_loja", "ano", "mes")
    .agg(F.count("*").alias("qtd_transacoes"),
         F.sum(F.when(F.col("cliente_identificado"), 1).otherwise(0)).alias("qtd_com_cpf"),
         F.round(F.sum(F.when(F.col("cliente_identificado"), F.col("valor_total_venda")).otherwise(0)), 2)
          .alias("receita_com_cpf"),
         F.round(F.sum(F.when(~F.col("cliente_identificado"), F.col("valor_total_venda")).otherwise(0)), 2)
          .alias("receita_sem_cpf"))
    .withColumn("qtd_sem_cpf", F.col("qtd_transacoes") - F.col("qtd_com_cpf"))
    .withColumn("pct_com_cpf", F.round(F.col("qtd_com_cpf") / F.col("qtd_transacoes") * 100, 2))
    .withColumn("pct_sem_cpf", F.round(100 - F.col("pct_com_cpf"), 2))
    .withColumn("periodo", F.make_date("ano", "mes", F.lit(1)))
    .withColumn("_periodo_ant", F.lag("periodo").over(w_loja))
    .withColumn("_pct_ant", F.lag("pct_com_cpf").over(w_loja))
    .withColumn("pct_com_cpf_mes_anterior",
                F.when(F.col("_periodo_ant") == F.add_months("periodo", -1), F.col("_pct_ant")))
    # variação em pontos percentuais: crescimento = sucesso do CRM
    .withColumn("variacao_pct_com_cpf_pp", F.round(F.col("pct_com_cpf") - F.col("pct_com_cpf_mes_anterior"), 2))
    .drop("_periodo_ant", "_pct_ant")
)
publicar(cpf_mes, "gold_cpf_identificado_loja_mes", ["id_loja", "ano", "mes"], ["ano"])


gold_cpf_identificado_loja_mes  (chave: id_loja, ano, mes)
  [Delta] gold_cpf_identificado_loja_mes: esquema alterado, tabela reescrita
  [SQL Server] squad3.gold_cpf_identificado_loja_mes: esquema alterado, recriada


In [0]:
# REGRA 9: Número de transações por caixa por dia (caixas sobrecarregados)
# Referência: média de transações por caixa/dia da mesma loja no mesmo mês.
w_loja_mes = Window.partitionBy("id_loja", "ano", "mes")
caixa_dia = (
    vendas.groupBy("data_venda", "ano", "mes", "id_loja", "id_caixa")
    .agg(F.count("*").alias("qtd_transacoes"),
         F.round(F.sum("valor_total_venda"), 2).alias("receita_total"))
    .withColumn("media_caixa_dia_loja_mes", F.round(F.avg("qtd_transacoes").over(w_loja_mes), 2))
    .withColumn("razao_vs_media", F.round(F.col("qtd_transacoes") / F.col("media_caixa_dia_loja_mes"), 2))
    .withColumn("e_sobrecarregado", F.col("razao_vs_media") >= FATOR_SOBRECARGA)
    .join(calendario.select("data_venda", "e_feriado", "e_ponto_facultativo", "tipo_dia"), "data_venda", "left")
)
publicar(caixa_dia, "gold_transacoes_caixa_dia", ["data_venda", "id_loja", "id_caixa"], ["ano", "mes"])


gold_transacoes_caixa_dia  (chave: data_venda, id_loja, id_caixa)
  [Delta] gold_transacoes_caixa_dia: esquema alterado, tabela reescrita
  [SQL Server] squad3.gold_transacoes_caixa_dia: esquema alterado, recriada


In [0]:
# REGRA 10: vendas por dia com flag e_feriado (feriado x dia normal)
vendas_dia = (
    vendas.groupBy("data_venda")
    .agg(F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
         F.count("*").alias("qtd_transacoes"),
         F.countDistinct("id_loja").alias("lojas_com_venda"),
         F.sum(F.when(F.col("cliente_identificado"), 1).otherwise(0)).alias("qtd_com_cpf"))
    .join(calendario, "data_venda", "right")          # mantém dias sem venda (ex.: Natal)
    .fillna(0, subset=["receita_total", "qtd_transacoes", "lojas_com_venda", "qtd_com_cpf"])
    .withColumn("ticket_medio", F.round(F.col("receita_total") / F.when(F.col("qtd_transacoes") > 0,
                                                                         F.col("qtd_transacoes")), 2))
    .withColumn("ano", F.year("data_venda"))
    .withColumn("mes", F.month("data_venda"))
)
publicar(vendas_dia, "gold_vendas_dia_feriado", ["data_venda"], ["ano"])


gold_vendas_dia_feriado  (chave: data_venda)
  [Delta] gold_vendas_dia_feriado: esquema alterado, tabela reescrita
  [SQL Server] squad3.gold_vendas_dia_feriado: esquema alterado, recriada


## 5. Validação

In [0]:
def contar_sqlserver(nome: str):
    try:
        return ler_sqlserver(f"SELECT COUNT(*) AS n FROM [{SQL_SCHEMA}].[{nome}]").first()["n"]
    except Exception as e:
        print(f"  [aviso] contagem no SQL Server falhou para {nome}: {str(e)[:120]}")
        return None


def ler_gold(nome):
    return ler_delta(f"{GOLD_BASE}/{nome}")


print(f"{'tabela':45s}{'delta':>10s}{'sql server':>12s}")
for nome in PUBLICADAS:
    n_delta = ler_gold(nome).count()
    n_sql = contar_sqlserver(nome) if GRAVAR_SQLSERVER else None
    status = "OK" if (n_sql is None and not GRAVAR_SQLSERVER) or n_delta == n_sql else "DIVERGENTE"
    print(f"{nome:45s}{n_delta:>10,}{(n_sql if n_sql is not None else '-'):>12}  {status}")

receita_silver = round(vendas.agg(F.sum("valor_total_venda")).first()[0], 2)
qtd_silver = vendas.count()
checks = {
    "receita regra 6": ler_gold("gold_receita_loja_mes").agg(F.sum("receita_total")).first()[0],
    "receita regra 7": ler_gold("gold_ticket_medio_loja_pagamento_ano").agg(F.sum("receita_total")).first()[0],
    "receita regra 9": ler_gold("gold_transacoes_caixa_dia").agg(F.sum("receita_total")).first()[0],
    "receita regra 10": ler_gold("gold_vendas_dia_feriado").agg(F.sum("receita_total")).first()[0],
}
print(f"\nReceita Silver (válida): {receita_silver:,.2f} | transações: {qtd_silver:,}")
for k, v in checks.items():
    print(f"  {k}: {v:,.2f} -> {'OK' if abs(round(v, 2) - receita_silver) < 0.05 else 'DIVERGENTE'}")
qtd_cpf = ler_gold("gold_cpf_identificado_loja_mes").agg(F.sum("qtd_transacoes")).first()[0]
print(f"  transações regra 8: {qtd_cpf:,} -> {'OK' if qtd_cpf == qtd_silver else 'DIVERGENTE'}")

sobrec = ler_gold("gold_transacoes_caixa_dia").filter("e_sobrecarregado")
print(f"\nCaixas/dia sobrecarregados (>= {FATOR_SOBRECARGA}x a média da loja no mês): {sobrec.count():,}")
display(sobrec.orderBy(F.desc("razao_vs_media")).limit(20))

tabela                                            delta  sql server
gold_dim_lojas                                       29          29  OK
gold_receita_loja_mes                               928         928  OK
gold_ticket_medio_loja_pagamento_ano                261         261  OK
gold_cpf_identificado_loja_mes                      928         928  OK
gold_transacoes_caixa_dia                       123,631      123631  OK
gold_vendas_dia_feriado                             974         974  OK

Receita Silver (válida): 137,214,981.41 | transações: 375,000
  receita regra 6: 137,214,981.41 -> OK
  receita regra 7: 137,214,981.41 -> OK
  receita regra 9: 137,214,981.41 -> OK
  receita regra 10: 137,214,981.41 -> OK
  transações regra 8: 375,000 -> OK

Caixas/dia sobrecarregados (>= 3.0x a média da loja no mês): 271


data_venda,ano,mes,id_loja,id_caixa,qtd_transacoes,receita_total,media_caixa_dia_loja_mes,razao_vs_media,e_sobrecarregado,e_feriado,e_ponto_facultativo,tipo_dia,gold_processed_at
2024-05-06,2024,5,15,5,11,4484.58,2.34,4.7,true,false,false,dia útil,2026-10-23T20:28:42.556Z
2026-03-01,2026,3,25,1,12,5469.31,2.76,4.35,true,false,false,fim de semana,2026-10-23T20:28:42.556Z
2025-01-05,2025,1,20,3,7,2478.03,1.66,4.22,true,false,false,fim de semana,2026-10-23T20:28:42.556Z
2025-03-24,2025,3,13,3,9,5860.77,2.14,4.21,true,false,false,dia útil,2026-10-23T20:28:42.556Z
2026-04-12,2026,4,11,3,8,2516.16,1.9,4.21,true,false,false,fim de semana,2026-10-23T20:28:42.556Z
2026-02-22,2026,2,15,4,8,1363.87,1.94,4.12,true,false,false,fim de semana,2026-10-23T20:28:42.556Z
2026-05-07,2026,5,13,2,9,3752.81,2.19,4.11,true,false,false,dia útil,2026-10-23T20:28:42.556Z
2026-07-20,2026,7,16,1,7,1790.21,1.72,4.07,true,false,false,dia útil,2026-10-23T20:28:42.556Z
2024-03-25,2024,3,18,4,7,2763.89,1.72,4.07,true,false,false,dia útil,2026-10-23T20:28:42.556Z
2025-05-16,2025,5,26,1,8,2396.5,1.98,4.04,true,false,false,dia útil,2026-10-23T20:28:42.556Z


## 6. Exportação CSV 


In [0]:
if EXPORTAR_CSV:
    if CSV_DIR.startswith("/Volumes/"):
        cat, sch, vol = CSV_DIR.split("/")[2:5]
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS {cat}.{sch}.{vol}")
        except Exception as e:
            print(f"[aviso] não foi possível criar o volume {cat}.{sch}.{vol}: {str(e).splitlines()[0][:120]}")
    os.makedirs("/tmp/gold_csv", exist_ok=True)
    for nome in PUBLICADAS:
        local = f"/tmp/gold_csv/{nome}.csv"
        (ler_gold(nome).drop("gold_processed_at").toPandas()
            .to_csv(local, index=False, sep=";", decimal=",", encoding="utf-8-sig"))
        try:
            shutil.copy(local, f"{CSV_DIR}/{nome}.csv")
            print(f"  salvo: {CSV_DIR}/{nome}.csv")
        except Exception as e:
            print(f"  [erro] {nome}: {e}")
    print("\nPronto. Baixe os arquivos pelo Catalog Explorer.")

  salvo: /Volumes/workspace/default/exportacoes/gold_dim_lojas.csv
  salvo: /Volumes/workspace/default/exportacoes/gold_receita_loja_mes.csv
  salvo: /Volumes/workspace/default/exportacoes/gold_ticket_medio_loja_pagamento_ano.csv
  salvo: /Volumes/workspace/default/exportacoes/gold_cpf_identificado_loja_mes.csv
  salvo: /Volumes/workspace/default/exportacoes/gold_transacoes_caixa_dia.csv
  salvo: /Volumes/workspace/default/exportacoes/gold_vendas_dia_feriado.csv

Pronto. Baixe os arquivos pelo Catalog Explorer.
